<a href="https://colab.research.google.com/github/RajeevKambhatla/longevity-drug-repurposing/blob/main/01_pick_cell_type.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/longevity_project'
for folder in ['data/raw', 'data/processed', 'results']:
    os.makedirs(f'{BASE}/{folder}', exist_ok=True)
print(os.listdir(BASE))

Mounted at /content/drive
['data', 'results']


In [3]:
!wget -nc -P /content/drive/MyDrive/longevity_project/data/raw https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_sig_info.txt.gz

File ‘/content/drive/MyDrive/longevity_project/data/raw/GSE92742_Broad_LINCS_sig_info.txt.gz’ already there; not retrieving.



In [4]:
import pandas as pd
sig = pd.read_csv(f'{BASE}/data/raw/GSE92742_Broad_LINCS_sig_info.txt.gz',
                  sep='\t', low_memory=False)
print(sig.shape)
sig.head()

(473647, 12)


,sig_id,pert_id,pert_iname,pert_type,cell_id,pert_dose,pert_dose_unit,pert_idose,pert_time,pert_time_unit,pert_itime,distil_id
0,AML001_CD34_24H:A05,DMSO,DMSO,ctl_vehicle,CD34,0.1,%,0.1 %,24,h,24 h,AML001_CD34_24H_X1_F1B10:A05
1,AML001_CD34_24H:A06,DMSO,DMSO,ctl_vehicle,CD34,0.1,%,0.1 %,24,h,24 h,AML001_CD34_24H_X3_F1B10:A06
2,AML001_CD34_24H:B05,DMSO,DMSO,ctl_vehicle,CD34,0.1,%,0.1 %,24,h,24 h,AML001_CD34_24H_X1_F1B10:B05|AML001_CD34_24H_X...
3,AML001_CD34_24H:B06,DMSO,DMSO,ctl_vehicle,CD34,0.1,%,0.1 %,24,h,24 h,AML001_CD34_24H_X3_F1B10:B06
4,AML001_CD34_24H:BRD-A03772856:0.37037,BRD-A03772856,BRD-A03772856,trt_cp,CD34,0.37037,µM,500 nM,24,h,24 h,AML001_CD34_24H_X1_F1B10:J04|AML001_CD34_24H_X...


In [5]:
sig = sig[sig['pert_type'] == 'trt_cp']
print('Signatures:', len(sig))
print('Drugs:', sig['pert_iname'].nunique())
print('Cell types:', sig['cell_id'].nunique())

Signatures: 205034
Drugs: 19811
Cell types: 71


In [6]:
known = ['sirolimus', 'rapamycin', 'everolimus', 'metformin', 'acarbose',
         'dasatinib', 'quercetin', 'fisetin', 'navitoclax',
         'resveratrol', 'spermidine', 'nicotinamide']
names = set(sig['pert_iname'].str.lower())
for d in known:
    print(d, d in names)

sirolimus True
rapamycin False
everolimus True
metformin True
acarbose True
dasatinib True
quercetin True
fisetin False
navitoclax True
resveratrol True
spermidine True
nicotinamide True


In [7]:
sorted(n for n in names if 'querc' in n)

['isoquercetin', 'quercetagetin', 'quercetin']

In [8]:
found = [d for d in known if d in names]
sig['name_lower'] = sig['pert_iname'].str.lower()
known_count = sig[sig['name_lower'].isin(found)].groupby('cell_id')['name_lower'].nunique()
total_count = sig.groupby('cell_id')['name_lower'].nunique()
table = pd.DataFrame({'known_drugs': known_count, 'total_drugs': total_count}).dropna()
table = table.sort_values(['known_drugs', 'total_drugs'], ascending=False)
table.head(10)

,known_drugs,total_drugs
cell_id,,
VCAP,9.0,15542
A549,9.0,12193
A375,9.0,11031
MCF7,9.0,10755
PC3,9.0,10661
HT29,9.0,10612
HCC515,9.0,4779
HA1E,9.0,4636
NPC,9.0,3345


In [9]:
table.to_csv(f'{BASE}/results/cell_type_coverage.csv')

import json
with open(f'{BASE}/results/config.json', 'w') as f:
    json.dump({'cell_type': 'VCAP'}, f)

print(table.loc['VCAP'])

known_drugs        9.0
total_drugs    15542.0
Name: VCAP, dtype: float64
